# Neuron art generator 🧠🍕
Procedural fluorescent-neuron images + a **"neuropizza"** neuron-lettering logo.
Every render saves a black-background PNG and a **transparent** PNG (ready for Canva).

Run the cells top to bottom. Change the settings in the cells marked ⚙️ and re-run them.

In [ ]:
%pip install -q pillow numpy scipy scikit-image

## 1. Setup (shared helpers)

In [ ]:
import math, random, os
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from scipy import ndimage as ndi
from skimage.morphology import skeletonize
from IPython.display import display


def finish(img, color=(40, 255, 90), glow=0.5, name="out", dark=None):
    """Takes a grayscale PIL image (white = neuron), adds glow + colour + noise,
    saves <name>.png (black bg) and <name>_transparent.png, returns the black version."""
    a = np.asarray(img, dtype=np.float32) / 255.0
    g1 = np.asarray(img.filter(ImageFilter.GaussianBlur(4)), np.float32) / 255
    g2 = np.asarray(img.filter(ImageFilter.GaussianBlur(26)), np.float32) / 255
    lum = np.clip(a + glow * (0.55 * g1 + 0.9 * g2), 0, 1)
    if dark is not None:   # darker patches (e.g. nucleus) applied AFTER glow so they stay visible
        lum = np.clip(lum - 0.45 * np.asarray(dark, np.float32) / 255, 0, 1)
    lum = np.clip(lum + np.random.normal(0, 0.012, lum.shape), 0, 1) ** 0.9

    col = np.array(color, np.float32) / 255
    rgb = lum[..., None] * col
    rgb += (np.clip(lum - 0.85, 0, 1) * 2.5)[..., None] * np.array([0.5, 0.2, 0.5], np.float32)
    rgb = np.clip(rgb, 0, 1)
    black = Image.fromarray((rgb * 255).astype(np.uint8))
    black.save(f"{name}.png", dpi=(300, 300))

    alpha = np.clip(lum * 1.25, 0, 1)
    bright = np.clip(col[None, None, :] * 0.85 + (np.clip(lum - 0.8, 0, 1) * 2.0)[..., None] * 0.5, 0, 1)
    Image.fromarray((np.dstack([bright, alpha]) * 255).astype(np.uint8), "RGBA").save(
        f"{name}_transparent.png", dpi=(300, 300))
    print(f"saved {name}.png and {name}_transparent.png  ({img.width}x{img.height})")
    return black


def blob(d, cx, cy, rx, ry, rot=0.0, irregular=0.18, taper=0.0, fill=255, n=120):
    """Organic lumpy blob instead of a perfect ellipse.
    irregular = how lumpy the outline is, taper = squeeze toward the +rot direction (teardrop)."""
    ks = [2, 3, 4, 5, 7, 9]
    amps = [random.uniform(0.3, 1) / k for k in ks]
    phs = [random.uniform(0, 6.283) for _ in ks]
    norm = sum(amps)
    pts = []
    for i in range(n):
        t = 2 * math.pi * i / n
        s = 1 + irregular * sum(a * math.sin(k * t + p) for a, k, p in zip(amps, ks, phs)) / norm * 1.8
        fwd = max(0.0, math.cos(t))
        x = math.cos(t) * rx * s * (1 + taper * 0.6 * fwd)
        y = math.sin(t) * ry * s * (1 - taper * fwd ** 2)
        pts.append((cx + x * math.cos(rot) - y * math.sin(rot),
                    cy + x * math.sin(rot) + y * math.cos(rot)))
    d.polygon(pts, fill=fill)


def preview(im, max_w=900):
    """Show a smaller copy in the notebook (files on disk stay full-size)."""
    s = max_w / im.width
    display(im.resize((max_w, int(im.height * s)), Image.LANCZOS))


## 2. The neuron

In [ ]:
def make_neuron(seed=7, W=3000, H=4000, soma_pos=(0.62, 0.20), soma_r=70,
                apical_angle=math.radians(105), n_basal=6, n_axons=6, spines=True,
                color=(40, 255, 90), glow=0.5, name="neuron"):
    random.seed(seed); np.random.seed(seed)
    img = Image.new("L", (W, H), 0)
    d = ImageDraw.Draw(img)
    sx, sy = soma_pos[0] * W, soma_pos[1] * H

    def seg(x0, y0, x1, y1, width, bright=255):
        d.line([(x0, y0), (x1, y1)], fill=bright, width=max(1, int(width)))
        r = width / 2
        if r >= 1:
            d.ellipse([x1 - r, y1 - r, x1 + r, y1 + r], fill=bright)

    def spine(x0, y0, x1, y1, width):
        if not spines or width > 5: return
        if random.random() < 0.18:
            a = math.atan2(y1 - y0, x1 - x0) + random.choice([-1, 1]) * math.radians(random.uniform(60, 110))
            L = random.uniform(4, 11)
            ex, ey = x1 + math.cos(a) * L, y1 + math.sin(a) * L
            d.line([(x1, y1), (ex, ey)], fill=200, width=1)
            r = random.uniform(1.2, 2.2)
            d.ellipse([ex - r, ey - r, ex + r, ey + r], fill=230)

    def grow(x, y, angle, length_left, width, depth, wander=0.22, branch_p=0.012, stiff=0.0, bias=None, flare=False):
        step = 7
        trav = 0
        while length_left > 0 and width > 0.6:
            angle += random.gauss(0, wander)
            if bias is not None:
                diff = (bias - angle + math.pi) % (2 * math.pi) - math.pi
                angle += diff * stiff
            nx, ny = x + math.cos(angle) * step, y + math.sin(angle) * step
            if not (-50 < nx < W + 50 and -50 < ny < H + 50): break
            fw = width * (1 + 1.1 * max(0.0, 1 - trav / 90)) if flare else width   # thick where it leaves the soma
            seg(x, y, nx, ny, fw); spine(x, y, nx, ny, width)
            trav += step
            x, y = nx, ny
            length_left -= step
            width *= 0.9985 if depth == 0 else 0.992
            if depth < 5 and random.random() < branch_p * (1 + depth * 0.4):
                spread = math.radians(random.uniform(25, 55)); sign = random.choice([-1, 1])
                grow(x, y, angle + sign * spread, length_left * random.uniform(0.5, 0.85),
                     width * 0.72, depth + 1, wander * 1.1, branch_p, stiff * 0.5, bias)
                angle -= sign * spread * 0.35

    # apical trunk + tuft + basal dendrites
    grow(sx, sy, apical_angle, H * 1.0, 16, 0, wander=0.05, branch_p=0.006, stiff=0.03, bias=apical_angle, flare=True)
    for _ in range(3):
        grow(sx, sy, apical_angle + math.pi + random.uniform(-0.6, 0.6),
             random.uniform(350, 600), 7, 1, wander=0.2, branch_p=0.012, flare=True)
    for i in range(n_basal):
        a = (i / n_basal) * 2 * math.pi + random.uniform(-0.3, 0.3)
        grow(sx, sy, a, random.uniform(350, 800), 9, 1, wander=0.25, branch_p=0.010, flare=True)
    # soma
    # soma: lumpy teardrop (narrow end toward the trunk), with a darker nucleus + mottled cytoplasm
    dark = Image.new("L", (W, H), 0); dk = ImageDraw.Draw(dark)
    blob(d, sx, sy, soma_r * 1.15, soma_r * 0.85, rot=apical_angle, irregular=0.16, taper=0.45, fill=235)
    for _ in range(6):                                   # brighter patches
        ox, oy = random.gauss(0, soma_r * 0.35), random.gauss(0, soma_r * 0.3)
        blob(d, sx + ox, sy + oy, soma_r * random.uniform(0.3, 0.5), soma_r * random.uniform(0.25, 0.4),
             rot=random.uniform(0, 6.28), irregular=0.25, fill=255)
    nx0, ny0 = sx + random.gauss(0, soma_r * 0.12), sy + random.gauss(0, soma_r * 0.12)
    blob(dk, nx0, ny0, soma_r * 0.50, soma_r * 0.42, rot=random.uniform(0, 6.28), irregular=0.1, fill=255)   # nucleus
    for _ in range(5):                                   # subtle mottling
        blob(dk, sx + random.gauss(0, soma_r * 0.4), sy + random.gauss(0, soma_r * 0.35),
             soma_r * 0.2, soma_r * 0.15, rot=random.uniform(0, 6.28), irregular=0.3, fill=70)
    dark = dark.filter(ImageFilter.GaussianBlur(soma_r * 0.12))
    # axons
    for _ in range(n_axons):
        x, y = random.choice([-10, W + 10]), random.uniform(0.3, 0.95) * H
        home = 0 if x < 0 else math.pi
        a = home + random.uniform(-0.15, 0.15)
        for _ in range(int(W / 8) + 50):
            a += random.gauss(0, 0.07) - (a - home) * 0.01
            nx, ny = x + math.cos(a) * 8, y + math.sin(a) * 8
            d.line([(x, y), (nx, ny)], fill=random.randint(120, 190), width=1)
            if random.random() < 0.004:
                ba = a + random.choice([-1, 1]) * 0.8; bx, by = nx, ny
                for _ in range(random.randint(8, 30)):
                    ba += random.gauss(0, 0.15)
                    ex, ey = bx + math.cos(ba) * 7, by + math.sin(ba) * 7
                    d.line([(bx, by), (ex, ey)], fill=90, width=1); bx, by = ex, ey
            x, y = nx, ny
    return finish(img, color, glow, name, dark=dark)


In [ ]:
# ⚙️ NEURON SETTINGS - change and re-run
neuron = make_neuron(
    seed=7,                       # any number = a different neuron
    W=3000, H=4000,               # size in px
    color=(40, 255, 90),          # (R,G,B): try (60,200,255) cyan, (255,80,200) magenta
    glow=0.5,                     # 0 = crisp, 1+ = dreamy
    soma_pos=(0.62, 0.20),        # where the cell body sits (fraction of canvas)
    apical_angle=math.radians(105),  # direction of the main trunk (90 = straight down)
    n_basal=6, n_axons=6, spines=True,
    name="neuron",
)
preview(neuron, 500)

## 3. The "neuropizza" logo

In [ ]:
# Put your own .ttf first in this list if you want a different font
FONTS = [
    "Poppins-Bold.ttf",
    "/System/Library/Fonts/Supplemental/Arial Rounded Bold.ttf",   # macOS
    "/System/Library/Fonts/Helvetica.ttc",
    "C:/Windows/Fonts/arialbd.ttf", "C:/Windows/Fonts/segoeuib.ttf",  # Windows
    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",        # Linux
]


def make_logo(text="neuropizza", seed=3, W=4000, H=1600, color=(40, 255, 90),
              text_fill=0.86, stroke=0.20, sprout_p=0.12, end_sprouts=2, soma=2.4,
              spines=True, wobble=7, glow=0.45, name="neuropizza"):
    random.seed(seed); np.random.seed(seed)

    # 1. text mask
    font_path = next((f for f in FONTS if os.path.exists(f)), None)
    if font_path is None: raise SystemExit("No font found - add a .ttf path to FONTS.")
    size = 500
    font = ImageFont.truetype(font_path, size)
    l, t, r, b = font.getbbox(text)
    size = int(size * (W * text_fill) / (r - l))
    font = ImageFont.truetype(font_path, size)
    l, t, r, b = font.getbbox(text)
    m = Image.new("L", (W, H), 0)
    ImageDraw.Draw(m).text(((W - (r - l)) / 2 - l, (H - (b - t)) / 2 - t), text, fill=255, font=font)
    mask = np.asarray(m) > 127

    # 2. skeleton + thickness
    skel = skeletonize(mask)
    dist = ndi.distance_transform_edt(mask)
    ys, xs = np.nonzero(skel)
    nb = ndi.convolve(skel.astype(np.uint8), np.ones((3, 3), np.uint8), mode="constant") - 1
    nbr = nb[ys, xs]
    ends = [(x, y) for x, y, n in zip(xs, ys, nbr) if n == 1]
    junctions = [(x, y) for x, y, n in zip(xs, ys, nbr) if n >= 3]

    # 3. draw
    img = Image.new("L", (W, H), 0)
    d = ImageDraw.Draw(img)

    def seg(x0, y0, x1, y1, width, bright=255):
        d.line([(x0, y0), (x1, y1)], fill=bright, width=max(1, int(width)))
        rr = width / 2
        if rr >= 1: d.ellipse([x1 - rr, y1 - rr, x1 + rr, y1 + rr], fill=bright)

    for x, y in zip(xs, ys):
        rad = max(3.0, dist[y, x] * stroke)
        d.ellipse([x - rad, y - rad, x + rad, y + rad], fill=255)

    def spine(x1, y1, ang):
        if spines and random.random() < 0.22:
            a = ang + random.choice([-1, 1]) * math.radians(random.uniform(60, 110))
            L = random.uniform(5, 13)
            ex, ey = x1 + math.cos(a) * L, y1 + math.sin(a) * L
            d.line([(x1, y1), (ex, ey)], fill=200, width=1)
            rr = random.uniform(1.4, 2.4)
            d.ellipse([ex - rr, ey - rr, ex + rr, ey + rr], fill=235)

    def grow(x, y, ang, length, width, depth, wander=0.3, branch_p=0.02):
        step = 5
        while length > 0 and width > 0.7:
            ang += random.gauss(0, wander)
            nx, ny = x + math.cos(ang) * step, y + math.sin(ang) * step
            if not (0 < nx < W and 0 < ny < H): break
            seg(x, y, nx, ny, width)
            if width < 5: spine(nx, ny, ang)
            x, y = nx, ny
            length -= step; width *= 0.985
            if depth < 3 and random.random() < branch_p:
                sgn = random.choice([-1, 1])
                grow(x, y, ang + sgn * math.radians(random.uniform(25, 55)),
                     length * random.uniform(0.5, 0.8), width * 0.75, depth + 1, wander, branch_p)

    def local_dir(x, y, rad=16):
        y0, y1, x0, x1 = max(0, y - rad), min(H, y + rad + 1), max(0, x - rad), min(W, x + rad + 1)
        py, px = np.nonzero(skel[y0:y1, x0:x1]); px = px + x0; py = py + y0
        if len(px) < 3: return random.uniform(0, 2 * math.pi), None
        cx, cy = px.mean(), py.mean()
        w, v = np.linalg.eigh(np.cov(np.vstack([px - cx, py - cy])))
        return math.atan2(v[1, 1], v[0, 1]), (x - cx, y - cy)

    for i in range(len(xs)):
        if random.random() < sprout_p / 28:
            x, y = int(xs[i]), int(ys[i])
            tang, _ = local_dir(x, y)
            ang = tang + random.choice([-1, 1]) * math.pi / 2 + random.gauss(0, 0.4)
            grow(x, y, ang, random.uniform(60, 200), max(2.2, dist[y, x] * stroke * 0.5), 1)

    for x, y in ends:
        tang, off = local_dir(x, y)
        base = math.atan2(off[1], off[0]) if off and (off[0] or off[1]) else random.uniform(0, 6.28)
        for _ in range(end_sprouts):
            grow(x, y, base + random.gauss(0, 0.7), random.uniform(100, 260),
                 max(2.8, dist[y, x] * stroke * 0.7), 1, wander=0.25, branch_p=0.025)

    if soma > 0:
        for x, y in ends + random.sample(junctions, min(len(junctions), 14)):
            rr = max(6, dist[y, x] * stroke * soma)
            blob(d, x, y, rr, rr * 0.85, rot=random.uniform(0, 6.28), irregular=0.22, fill=255)

    # 4. wobble
    if wobble > 0:
        a = np.asarray(img, dtype=np.float32)
        def noise(sc):
            n = np.random.randn(H // sc + 2, W // sc + 2).astype(np.float32)
            return np.asarray(Image.fromarray(n).resize((W, H), Image.BICUBIC)) * wobble
        yy, xx = np.mgrid[0:H, 0:W].astype(np.float32)
        a = ndi.map_coordinates(a, [yy + noise(160), xx + noise(160)], order=1, mode="constant")
        img = Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))

    return finish(img, color, glow, name)


In [ ]:
# ⚙️ LOGO SETTINGS - change and re-run
logo = make_logo(
    text="neuropizza",            # any word
    seed=3,                       # new number = new random dendrites
    color=(40, 255, 90),          # try (255,120,40) for pizza-orange
    stroke=0.20,                  # letter thickness (0.2 thin ... 0.6 chunky)
    sprout_p=0.12,                # dendrites along the letters
    end_sprouts=2,                # dendrites at each letter end
    soma=2.4,                     # blob size at ends/junctions (0 = none)
    wobble=7,                     # organic wobble in px (0 = perfect)
    glow=0.45,
    name="neuropizza",
)
preview(logo, 1000)

## 4. Use in Canva
Upload the `*_transparent.png` files (they're saved next to this notebook) and place them over any background.
Glow looks best on dark/saturated backgrounds; on light backgrounds set `glow=0`.